# 01 — Data Exploration & Quality Check

Tujuan: memvalidasi struktur CSV, label, missing values, duplikasi, distribusi sentimen, aplikasi, dan panjang ulasan. Jangan melakukan perubahan dataset permanen pada notebook EDA.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import RAW_DATA_PATH
from src.data_loader import load_dataset

df = load_dataset(RAW_DATA_PATH)
print("Shape:", df.shape)
display(df.head())
print("\nDtypes:")
display(df.dtypes.to_frame("dtype"))


## 1. Missing values and duplicates

Periksa duplikasi baris penuh, teks review yang sama, dan teks sama dengan label sentimen berbeda. Jangan langsung menghapus sebelum tim mendokumentasikan keputusan.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing.to_frame("missing_count"))
print("Exact duplicate rows:", int(df.duplicated().sum()))
if "review" in df.columns:
    review_norm = df["review"].fillna("").astype(str).str.lower().str.strip()
    print("Duplicate normalized review texts:", int(review_norm.duplicated().sum()))
    conflict_check = (
        pd.DataFrame({"review_norm": review_norm, "sentiment": df["sentiment"].astype(str).str.strip().str.lower()})
        .groupby("review_norm")["sentiment"].nunique()
    )
    print("Identical texts with conflicting sentiment labels:", int((conflict_check > 1).sum()))


## 2. Label distribution

Distribusi ini menentukan kebutuhan evaluasi per kelas dan class weighting.

In [ ]:
display(df["sentiment"].value_counts(dropna=False).to_frame("count"))
ax = sns.countplot(data=df, x="sentiment", order=df["sentiment"].value_counts().index)
ax.set_title("Sentiment Distribution")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Number of reviews")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 3. Application distribution and review length

Gunakan atribut aplikasi untuk memahami cakupan data, bukan sebagai fitur model utama.

In [ ]:
if "app_name" in df.columns:
    display(df["app_name"].value_counts(dropna=False).to_frame("count"))
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, y="app_name", order=df["app_name"].value_counts().index)
    plt.title("Reviews by Application")
    plt.tight_layout()
    plt.show()

df["review_length_words"] = df["review"].fillna("").astype(str).str.split().str.len()
display(df["review_length_words"].describe().to_frame("review_length_words"))
plt.figure(figsize=(8, 4))
sns.histplot(df["review_length_words"], bins=40)
plt.title("Review Length Distribution")
plt.xlabel("Words per review")
plt.tight_layout()
plt.show()


## 4. EDA conclusions

Catat jumlah data aktual, label yang ditemukan, proporsi setiap kelas, missing values, duplikasi, konflik label, dan keputusan preprocessing. Jangan menyalin angka dokumentasi sebelum memeriksa CSV yang diunduh.